In [28]:
from typing import TypedDict

class Agentstate(TypedDict):
    question:str
    tool:str
    tool_output:str
    answer:str    

In [29]:
import boto3
import json
import os
from dotenv import load_dotenv

load_dotenv("myenv.env")

# AWS Credentials
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION")


In [30]:
from langchain_aws import ChatBedrockConverse

llm = ChatBedrockConverse(
    model="amazon.nova-micro-v1:0",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)

In [31]:
import sqlite3
con = sqlite3.connect("employee.db")
 
cursor=con.cursor()

cursor.execute(""" CREATE TABLE IF NOT EXISTS employee( 
              employee_id INTEGER,
              employee_name TEXT,
              manager TEXT
              )
            """)
cursor.execute("DELETE FROM employee")
cursor.execute(""" INSERT INTO employee VALUES (99, 'vikas' ,'sam Altman')
               """)

cursor.execute(""" INSERT INTO employee VALUES (100, 'mike' ,'Satya Nadella')
               """)

cursor.execute(""" INSERT INTO employee VALUES (101, 'akash' ,'Sridhar Vembu')
               """)
cursor.execute(""" INSERT INTO employee VALUES (102, 'nikki' ,'Sundar pichai')
               """)

con.commit()
con.close()
print("database ready")

database ready


In [32]:
def db_query(employee_id):
    con = sqlite3.connect("employee.db")
    cursor= con.cursor()
    cursor.execute("""
                SELECT employee_name, manager 
                FROM employee WHERE employee_id=? """, (int(employee_id),)
                )
    result =  cursor.fetchone()

    con.close()

    if result:
        return  (
            f"Employee:{result[0]}, "
            f"Manager:{result[1]}"
        )
    return "Employee not found"

In [33]:
from ddgs import DDGS
def web_search(query):
    with DDGS() as ddgs:
        result =list(ddgs.text(query, max_results=5))

    print(result)
    
    if len(result)==0:
        return "NO result found"
    return result[0]["body"]
    # print(result)

In [52]:
def decide_tool(state):

    q = state["question"].lower()

    if "employee" in q or "manager" in q:
        return {"tool":"db"}

    return {"tool":"web"}

In [ ]:
# def decide_tool(state):

#     prompt = f"""
#     Route the question.

#     db = employee lookup
#     sql = aggregate employee queries
#     web = internet search

#     Question:
#     {state['question']}

#     Return ONLY one word:

#     db
#     sql
#     web
#     """

#     response = llm.invoke(prompt)

#     text = response.content.lower()

#     if "db" in text:
#         tool = "db"

#     elif "sql" in text:
#         tool = "sql"

#     else:
#         tool = "web"

#     return {
#         "tool": tool
#     }

In [53]:
def router(state):
    return state["tool"]

In [54]:
import re

def db_node(state):

    question = state["question"]

    matches = re.findall(r"\d+", question)

    if not matches:
        return {
            "tool_output": "Please provide a valid employee ID."
        }

    emp_id = matches[0]

    result = db_query(emp_id)

    return {
        "tool_output": result
    }

In [55]:
import sqlite3

def sql_node(state):

    con = sqlite3.connect("employee.db")
    cursor = con.cursor()

    cursor.execute(
        "SELECT COUNT(*) FROM employee"
    )

    count = cursor.fetchone()[0]

    con.close()

    return {
        "tool_output": f"Total employees: {count}"
    }

In [70]:

def web_node(state):

    print("WEB NODE ENTERED")

    result = web_search(state["question"])

    print("WEB RESULT RECEIVED")

    return {
        "tool_output": str(result)
    }

In [71]:
def answer_node(state):
    return {
        "answer":state["tool_output"]
    }

In [72]:
from langgraph.graph import StateGraph, END
builder= StateGraph(Agentstate)

builder.add_node("decide", decide_tool)
builder.add_node("db", db_node)
builder.add_node("sql", sql_node)
builder.add_node("web", web_node)
builder.add_node("answer", answer_node)

In [73]:
builder.add_conditional_edges(
    "decide",
    router,
    {
        "db": "db",
        "sql":"sql",
        "web": "web"
    }
)

In [74]:
builder.add_edge("db","answer")
builder.add_edge("sql", "answer")
builder.add_edge("web", "answer")

builder.add_edge("answer", END)

In [76]:
builder.set_entry_point("decide")

In [78]:
graph=builder.compile()

In [79]:
response = graph.invoke({
    "question": "Who manages employee 100?"
})

print(response)

{'question': 'Who manages employee 100?', 'tool': 'db', 'tool_output': 'Employee:mike, Manager:Satya Nadella', 'answer': 'Employee:mike, Manager:Satya Nadella'}


In [80]:

response = graph.invoke({
    "question":"who manages employee 99 and which is his g currently working company ?"
})

print(response["answer"])

Employee:vikas, Manager:sam Altman
